# Per-meet engagement
Drill into a single meet day. Pick the meet from the dropdown below — it is populated from the meets actually seen in telemetry (last `LOOKBACK_DAYS` days). The query window is then derived from the meet's own date, so meets older than any relative lookback still return data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('_lib'))
from engagement_query import (run_kql, per_meet_query, timespan_for_meet_day,
                              meet_dropdown, workspace_id_from_env)
import pandas as pd

LOOKBACK_DAYS = 180  # how far back the meet dropdown searches
WS = workspace_id_from_env()
picker = meet_dropdown(WS, days=LOOKBACK_DAYS)
picker

In [ ]:
MEET_ID, PI_LOCAL_DATE = picker.value
# The timespan comes from the meet date itself (with a 1-day margin each
# side), not a relative "last N days" window — old meets work too.
df = run_kql(WS, per_meet_query(MEET_ID, PI_LOCAL_DATE),
             timespan=timespan_for_meet_day(PI_LOCAL_DATE))
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"{len(df)} rows for {MEET_ID} on {PI_LOCAL_DATE}")
df.head()

## Viewing sessions (5-min gap rule)

In [ ]:
sessions = (df.groupby(['viewer_id', 'session_idx'])
             .agg(start=('timestamp', 'min'),
                  end=('timestamp', 'max'),
                  events=('ev', 'count'))
             .reset_index())
sessions['duration_s'] = (sessions['end'] - sessions['start']).dt.total_seconds()
sessions.describe()

## Drop-off by event type

In [ ]:
views = df[df['ev'] == 'viewer_event_view']
drop = (views.groupby(['stroke_name', 'age_group', 'gender'])
             .agg(unique_viewers=('viewer_id', 'nunique'),
                  views=('viewer_id', 'count'))
             .reset_index()
             .sort_values('unique_viewers', ascending=False))
drop.head(30)

## Concurrent-viewer peak (1-minute buckets)

In [ ]:
hb = df[df['ev'] == 'viewer_heartbeat'].copy()
hb['bucket'] = hb['timestamp'].dt.floor('1min')
peak = hb.groupby('bucket')['viewer_id'].nunique()
peak.plot(title='Concurrent viewers per minute');

## Message board engagement
`viewer_message_board_view` fires when the message board overlay turns on/off or rotates pages (`mb_active`, `mb_page_index`), so viewing time on the board is attributed separately from race events.

In [ ]:
mb = df[df['ev'] == 'viewer_message_board_view']
mb_on = mb[mb['mb_active'] == True]
print(f"board activations seen: {len(mb_on)}, unique viewers: {mb_on['viewer_id'].nunique()}")
mb_on.groupby('mb_page_index')['viewer_id'].nunique().rename('unique_viewers').to_frame()

## LCP and connection-type breakdown

In [ ]:
loads = df[df['ev'] == 'viewer_page_load']
loads['effective_type'].value_counts()

In [ ]:
lcp = df[df['ev'].isin(['viewer_hidden', 'viewer_unload'])]['lcp_ms']
lcp = lcp[lcp > 0]
lcp.describe()